# AI Talent Data Exploration

This notebook shows you how to explore and combine data from multiple sources to create your AI talent list.

## Data Sources:
- **CSRankings**: Faculty members (34,325 records)
- **OpenAlex Cache**: Publication and author data from OpenAlex API
- **DBLP**: AI conference papers

## Goal:
Extract a comprehensive list of AI researchers with their:
- Names
- Affiliations
- Publication counts
- Career mobility events
- Institution types (academia vs industry)

In [ ]:
import pandas as pd
import json
from pathlib import Path
from collections import defaultdict

print("✓ Imports loaded")

## 1. Load CSRankings Faculty Data

CSRankings provides a curated list of CS faculty with their affiliations and identifiers.

In [ ]:
# Load CSRankings data
csrankings = pd.read_parquet("data/interim/csrankings.parquet")

print(f"Total CSRankings Faculty: {len(csrankings):,}")
print(f"\nColumns: {list(csrankings.columns)}")
print(f"\nFirst 10 faculty members:")
csrankings.head(10)

In [ ]:
# Analyze CSRankings data
print("=== CSRankings Data Summary ===")
print(f"\nUnique affiliations: {csrankings['affiliation'].nunique():,}")
print(f"\nTop 15 institutions by faculty count:")
print(csrankings['affiliation'].value_counts().head(15))

## 2. Explore Cached OpenAlex Data

OpenAlex API responses are cached in `data/raw/openalex/`. Each directory contains JSON responses with author and publication data.

In [ ]:
# Check cached OpenAlex data
openalex_cache = Path("data/raw/openalex")
cache_dirs = [d for d in openalex_cache.iterdir() if d.is_dir()]

print(f"OpenAlex cache directories: {len(cache_dirs)}")
print(f"\nSample cache directory: {cache_dirs[0].name if cache_dirs else 'None'}")

# Look at one cached response to understand structure
if cache_dirs:
    sample_dir = cache_dirs[0]
    cache_files = list(sample_dir.glob("*.json"))
    print(f"Files in sample directory: {len(cache_files)}")
    
    if cache_files:
        with open(cache_files[0]) as f:
            sample_response = json.load(f)
        print(f"\nSample response keys: {list(sample_response.keys())}")

## 3. Parse OpenAlex Cache to Extract Authors

OpenAlex responses contain works with authorship data. Let's extract all unique authors.

In [ ]:
def extract_authors_from_openalex_cache(cache_dir, max_files=100):
    """
    Extract author data from cached OpenAlex responses.
    
    Args:
        cache_dir: Path to OpenAlex cache directory
        max_files: Maximum number of cache files to process (for testing)
    
    Returns:
        DataFrame with author data
    """
    authors = []
    works_count = 0
    files_processed = 0
    
    for cache_subdir in cache_dir.iterdir():
        if not cache_subdir.is_dir():
            continue
            
        for cache_file in cache_subdir.glob("*.json"):
            if files_processed >= max_files:
                break
                
            try:
                with open(cache_file) as f:
                    response = json.load(f)
                
                # OpenAlex returns a list of works
                if "results" in response:
                    works = response["results"]
                else:
                    works = [response] if isinstance(response, dict) else []
                
                for work in works:
                    if not isinstance(work, dict):
                        continue
                    
                    works_count += 1
                    work_id = work.get("id", "")
                    pub_year = work.get("publication_year")
                    
                    # Extract authorships
                    for authorship in work.get("authorships", []):
                        author = authorship.get("author", {})
                        institutions = authorship.get("institutions", [])
                        
                        author_id = author.get("id", "")
                        author_name = author.get("display_name", "")
                        
                        # Create one row per institution affiliation
                        if institutions:
                            for inst in institutions:
                                authors.append({
                                    "author_openalex_id": author_id,
                                    "author_name": author_name,
                                    "work_id": work_id,
                                    "publication_year": pub_year,
                                    "institution_name": inst.get("display_name", ""),
                                    "institution_id": inst.get("id", ""),
                                    "country_code": inst.get("country_code", ""),
                                })
                        else:
                            # No institution listed
                            authors.append({
                                "author_openalex_id": author_id,
                                "author_name": author_name,
                                "work_id": work_id,
                                "publication_year": pub_year,
                                "institution_name": None,
                                "institution_id": None,
                                "country_code": None,
                            })
                
                files_processed += 1
                
            except Exception as e:
                print(f"Error processing {cache_file}: {e}")
                continue
        
        if files_processed >= max_files:
            break
    
    print(f"Processed {files_processed} cache files")
    print(f"Found {works_count} works")
    print(f"Extracted {len(authors)} authorship records")
    
    return pd.DataFrame(authors)

# Extract authors from first 50 cache files (for testing)
print("Extracting authors from OpenAlex cache (first 50 files)...")
openalex_authors = extract_authors_from_openalex_cache(openalex_cache, max_files=50)

print(f"\nTotal authorship records: {len(openalex_authors):,}")
print(f"Unique authors: {openalex_authors['author_openalex_id'].nunique():,}")
print(f"\nSample data:")
openalex_authors.head(10)

## 4. Aggregate to Person-Level

Convert authorship records to a unique list of people.

In [ ]:
# Aggregate by author
people = openalex_authors.groupby('author_openalex_id').agg({
    'author_name': 'first',
    'work_id': 'nunique',  # Count unique publications
    'institution_name': lambda x: x.mode()[0] if len(x.mode()) > 0 else None,  # Most common affiliation
    'country_code': lambda x: x.mode()[0] if len(x.mode()) > 0 else None,
    'publication_year': ['min', 'max']  # Career span
}).reset_index()

# Flatten column names
people.columns = [
    'openalex_id', 'name', 'num_publications', 'primary_affiliation', 
    'country', 'first_publication_year', 'last_publication_year'
]

print(f"Total unique AI researchers (from sample): {len(people):,}")
print(f"\nTop 10 by publication count:")
people.nlargest(10, 'num_publications')[['name', 'num_publications', 'primary_affiliation']]

## 5. Match with CSRankings

See which OpenAlex authors are in CSRankings (known faculty).

In [ ]:
# Simple name-based matching (not perfect, but gives an idea)
csrankings_names = set(csrankings['name'].str.lower().str.strip())
people['in_csrankings'] = people['name'].str.lower().str.strip().isin(csrankings_names)

print(f"Authors matched to CSRankings: {people['in_csrankings'].sum():,}")
print(f"Authors NOT in CSRankings: {(~people['in_csrankings']).sum():,}")
print(f"\nMatching rate: {people['in_csrankings'].mean():.1%}")

print("\n=== Sample: Authors in CSRankings ===")
people[people['in_csrankings']].head(10)

In [ ]:
print("\n=== Sample: Authors NOT in CSRankings (industry, postdocs, students) ===")
people[~people['in_csrankings']].nlargest(10, 'num_publications')[[
    'name', 'num_publications', 'primary_affiliation', 'country'
]]

## 6. Analyze Institution Types

Let's categorize institutions as academia vs industry.

In [ ]:
# Simple heuristic to identify industry affiliations
industry_keywords = [
    'google', 'microsoft', 'meta', 'facebook', 'apple', 'amazon', 
    'deepmind', 'openai', 'anthropic', 'nvidia', 'intel', 'ibm',
    'alibaba', 'tencent', 'baidu', 'bytedance', 'huawei'
]

def classify_institution(inst_name):
    if pd.isna(inst_name):
        return 'unknown'
    inst_lower = inst_name.lower()
    for keyword in industry_keywords:
        if keyword in inst_lower:
            return 'industry'
    # Check for university/college keywords
    if any(word in inst_lower for word in ['university', 'college', 'institute of technology']):
        return 'academia'
    return 'other'

people['institution_type'] = people['primary_affiliation'].apply(classify_institution)

print("Institution type distribution:")
print(people['institution_type'].value_counts())

print("\n=== Top Industry Researchers ===")
industry_researchers = people[people['institution_type'] == 'industry']
industry_researchers.nlargest(15, 'num_publications')[[
    'name', 'num_publications', 'primary_affiliation'
]]

## 7. Export Combined Data

Save the combined talent list for further analysis.

In [ ]:
# Export to CSV
output_file = "data/interim/ai_talent_sample.csv"
people.to_csv(output_file, index=False)
print(f"Exported {len(people):,} researchers to {output_file}")

# Also export industry-only
industry_file = "data/interim/ai_talent_industry_sample.csv"
industry_researchers.to_csv(industry_file, index=False)
print(f"Exported {len(industry_researchers):,} industry researchers to {industry_file}")

## 8. Full Pipeline vs Manual Approach

**What this notebook does:**
- Extracts author data from cached OpenAlex responses
- Provides a sample of ~50 files (out of 256 total)
- Simple name-based matching with CSRankings
- Basic institution type classification

**What the full pipeline does better:**
- Processes ALL 256 cache directories (~23,100 works)
- Uses `data/orgs.csv` for accurate institution mapping
- Calculates fractional credit (1/n_institutions) to avoid double-counting
- Derives mobility events (academia→industry transitions)
- Validates mobility against CSRankings departure records
- Provides QA outputs for disambiguation and unmatched institutions

**Next step:** Run the full pipeline to get the complete dataset!

## 9. Process ALL Cache Files (Optional)

To process all 256 cache directories instead of just 50, run this cell (will take ~2-5 minutes):

In [ ]:
# Uncomment to process ALL cache files
# print("Processing ALL OpenAlex cache files...")
# openalex_authors_full = extract_authors_from_openalex_cache(openalex_cache, max_files=10000)
# print(f"\nTotal authorship records: {len(openalex_authors_full):,}")
# print(f"Unique authors: {openalex_authors_full['author_openalex_id'].nunique():,}")